# 03 — Modelos Baseline TF-IDF

Regresión Logística, LinearSVC y MultinomialNB como referencia.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from discurso_odio.config import load_settings
from discurso_odio.data.loaders import load_all_splits, load_split, get_class_weights
from discurso_odio.paths import REPORTS_FIGURES, REPORTS_METRICS
from discurso_odio.utils.plotting import setup_plot_style, get_label_names, get_class_palette
from discurso_odio.utils.seed import set_seed

settings = load_settings()
set_seed(settings['project']['seed'])
setup_plot_style()
LABELS = get_label_names()
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)
REPORTS_METRICS.mkdir(parents=True, exist_ok=True)
print('Proyecto:', settings['project']['name'])

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score
import joblib
from discurso_odio.paths import MODELS_DIR



In [ ]:
train = load_split('train')
test = load_split('test')
X_train, y_train = train['texto_modelo'], train['label']
X_test, y_test = test['texto_modelo'], test['label']
print(f'Train: {len(X_train)} | Test: {len(X_test)}')


In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=10000, sublinear_tf=True)
models = {
    'LogisticRegression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'LinearSVC': LinearSVC(class_weight='balanced', random_state=42),
    'MultinomialNB': MultinomialNB(),
}
results = {}
for name, clf in models.items():
    pipe = Pipeline([('tfidf', vectorizer), ('clf', clf)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    f1 = f1_score(y_test, preds, average='macro')
    results[name] = {'f1_macro': f1, 'predictions': preds}
    print(f'{name}: F1-macro = {f1:.4f}')
    print(classification_report(y_test, preds, target_names=LABELS))
    joblib.dump(pipe, MODELS_DIR / f'baseline_{name.lower()}.joblib')


In [ ]:
results_df = pd.DataFrame({k: {'f1_macro': v['f1_macro']} for k, v in results.items()}).T
results_df.to_csv(REPORTS_METRICS / 'baselines_tfidf.csv')
results_df
